In [3]:

"""
=========================================================================================
PROJECT: LANDSLIDE FORECASTING - TCN ULTIMATE (Balanced & Multi-Horizon)
=========================================================================================
[IMPROVEMENTS]
1. Advanced Balancing: Oversample Critical/Warning + Downsample Normal
2. TCN Architecture: Powerful Sequence Modeling
3. Multi-Horizon: Forecast 0m to 24h
"""

import os
import glob
import time
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.utils import resample

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Conv1D, GlobalAveragePooling1D, Add, Activation, SpatialDropout1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

import warnings
warnings.filterwarnings('ignore')

# ====================================================================
# 1. CONFIGURATION
# ====================================================================
class Config:
    DATA_DIR = "./data/"
    MODEL_DIR = "./model/tcn_balanced/"
    SCALER_PATH = "./model/tcn_bal_scaler.save"
    
    HORIZONS = [0, 30, 60, 180, 360, 720, 1440]
    SEQUENCE_LENGTH = 60
    TEST_KEYWORDS = ['label_for_dev108_test_prepared']
    
    RAW_COLS = ['rain', 'soil', 'temp', 'humi', 'geo']
    LABEL_COL = 'label'
    LABEL_MAP = {'normal': 0, 'warning': 1, 'critical': 2}
    
    # --- BALANCING SETTINGS ---
    TARGET_SAMPLES_PER_CLASS = 5000 # ปรับจำนวนตัวอย่างที่ต้องการต่อคลาส (ทั้ง Over และ Down)
    
    # Training
    BATCH_SIZE = 64
    EPOCHS = 100
    LEARNING_RATE = 2e-3
    
    # Class Weights (ยังคงไว้เพื่อความปลอดภัย 2 ชั้น)
    CLASS_WEIGHTS = {0: 1.0, 1: 3.0, 2: 5.0} # ลดน้ำหนักลงหน่อยเพราะเรา Balance แล้ว

cfg = Config()
if not os.path.exists(cfg.MODEL_DIR): os.makedirs(cfg.MODEL_DIR)
np.random.seed(42); tf.random.set_seed(42); random.seed(42)

def log(msg): print(f"[INFO] {time.strftime('%H:%M:%S')} - {msg}")
print("✅ Config Loaded.")

# ====================================================================
# 2. LOAD & FEATURE ENGINEERING
# ====================================================================
def load_data():
    log("Scanning Data...")
    files = glob.glob(os.path.join(cfg.DATA_DIR, "*.csv"))
    if not files: 
        files = glob.glob(os.path.join(cfg.DATA_DIR, "**/*.csv"), recursive=True)
    
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f)
            df.columns = [str(c).lower().strip().replace('.1', '') for c in df.columns]
            rename_map = {'temperature':'temp', 'hum':'humi', 'humidity':'humi', 'devid':'devID', 'time':'timestamp', 'date':'timestamp'}
            new_cols = {c: rename_map[c] for c in df.columns if c in rename_map}
            if new_cols: df = df.rename(columns=new_cols)
            
            if 'devID' in df.columns: 
                df['devID'] = df['devID'].astype(str).str.extract('(\d+)')[0].astype(float).fillna(0).astype(int)
            if 'timestamp' in df.columns: 
                df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
            
            # --- แก้ไขตรงนี้ (เคาะบรรทัดลงมา) ---
            for c in cfg.RAW_COLS: 
                if c not in df.columns: 
                    df[c] = 0.0
            # ---------------------------------
            
            filename = os.path.basename(f)
            df['is_test'] = any(k in filename for k in cfg.TEST_KEYWORDS)
            dfs.append(df)
            print(f"   + Loaded: {filename} [{'TEST' if df['is_test'].iloc[0] else 'TRAIN'}]")
        except Exception as e:
            pass # ข้ามไฟล์ที่อ่านไม่ได้
            
    return pd.concat(dfs, ignore_index=True) if dfs else None

def generate_advanced_features(df):
    # ... (ส่วนนี้เหมือนเดิม) ...
    log("Generating Features...")
    df = df.sort_values(['devID', 'timestamp']).reset_index(drop=True)
    df_list = []
    
    for dev, g in df.groupby('devID'):
        g = g.set_index('timestamp')
        g = g[~g.index.duplicated(keep='first')]
        if len(g) > 0: g = g.resample('1T').asfreq()
        g[cfg.RAW_COLS] = g[cfg.RAW_COLS].interpolate(limit_direction='both').fillna(0)
        
        g['rain_1h'] = g['rain'].rolling(60, min_periods=1).sum()
        g['soil_rate'] = g['soil'].diff().fillna(0)
        g['soil_acc']  = g['soil'].diff().diff().fillna(0)
        g['rain_6h'] = g['rain'].rolling(360, min_periods=1).sum()
        g['soil_trend_6h'] = g['soil'] - g['soil'].shift(360).fillna(method='bfill')
        g['rain_24h'] = g['rain'].rolling(1440, min_periods=1).sum()
        g['soil_max_24h'] = g['soil'].rolling(1440, min_periods=1).max()
        g['rain_x_soil'] = g['rain'] * g['soil']
        g['rain_x_geo']  = g['rain'] * g['geo'].abs()
        
        if cfg.LABEL_COL in g.columns:
            g[cfg.LABEL_COL] = g[cfg.LABEL_COL].fillna('normal').astype(str).str.lower().str.strip()
            g[cfg.LABEL_COL] = g[cfg.LABEL_COL].map(cfg.LABEL_MAP).fillna(0).astype(int)
        else: g[cfg.LABEL_COL] = 0
        
        g['is_test'] = g['is_test'].fillna(method='ffill').fillna(method='bfill')
        g['devID'] = dev
        df_list.append(g.reset_index())
        
    return pd.concat(df_list, ignore_index=True).fillna(0)

df_train_raw = load_data()
df_proc = generate_advanced_features(df_train_raw)

# ====================================================================
# 3. DATA PREPARATION & BALANCING (NEW!)
# ====================================================================
def create_sequences(df, feature_cols, horizon):
    Xs, ys, is_tests = [], [], []
    for dev, g in df.groupby('devID'):
        data = g[feature_cols].values
        labels = g[cfg.LABEL_COL].shift(-horizon).values
        test_flag = g['is_test'].values
        
        valid_len = len(g) - horizon
        if valid_len < cfg.SEQUENCE_LENGTH: continue
        
        for i in range(valid_len - cfg.SEQUENCE_LENGTH):
            Xs.append(data[i : i+cfg.SEQUENCE_LENGTH])
            ys.append(labels[i+cfg.SEQUENCE_LENGTH])
            is_tests.append(test_flag[i+cfg.SEQUENCE_LENGTH])
            
    return np.array(Xs), np.array(ys), np.array(is_tests)

def balance_data(X, y):
    """
    ปรับสมดุลข้อมูล (Both Oversampling & Downsampling)
    """
    print(f"   Original counts: {Counter(y)}")
    X_bal, y_bal = [], []
    
    for cls in np.unique(y):
        # หา Index ของแต่ละคลาส
        indices = np.where(y == cls)[0]
        count = len(indices)
        
        # ถ้าข้อมูลเยอะกว่าเป้าหมาย -> Downsample (สุ่มออก)
        # ถ้าข้อมูลน้อยกว่าเป้าหมาย -> Oversample (สุ่มเพิ่ม)
        replace = count < cfg.TARGET_SAMPLES_PER_CLASS
        
        chosen_indices = np.random.choice(
            indices, 
            cfg.TARGET_SAMPLES_PER_CLASS, 
            replace=replace
        )
        
        X_bal.append(X[chosen_indices])
        y_bal.append(y[chosen_indices])
        
    X_bal = np.concatenate(X_bal)
    y_bal = np.concatenate(y_bal)
    
    # Shuffle
    perm = np.random.permutation(len(X_bal))
    return X_bal[perm], y_bal[perm]

# ====================================================================
# 4. TCN MODEL ARCHITECTURE
# ====================================================================
def residual_block(x, filters, kernel_size, dilation_rate):
    prev_x = x
    x = Conv1D(filters=filters, kernel_size=kernel_size, dilation_rate=dilation_rate, padding='causal')(x)
    x = BatchNormalization()(x); x = Activation('relu')(x); x = SpatialDropout1D(0.3)(x)
    x = Conv1D(filters=filters, kernel_size=kernel_size, dilation_rate=dilation_rate, padding='causal')(x)
    x = BatchNormalization()(x); x = Activation('relu')(x); x = SpatialDropout1D(0.3)(x)
    if prev_x.shape[-1] != filters: prev_x = Conv1D(filters=filters, kernel_size=1, padding='same')(prev_x)
    return Add()([x, prev_x])

def build_tcn_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = residual_block(inputs, filters=64, kernel_size=3, dilation_rate=1)
    x = residual_block(x, filters=64, kernel_size=3, dilation_rate=2)
    x = residual_block(x, filters=64, kernel_size=3, dilation_rate=4)
    x = residual_block(x, filters=64, kernel_size=3, dilation_rate=8)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='relu')(x); x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs, outputs, name="TCN_Forecaster")
    model.compile(optimizer=Adam(learning_rate=cfg.LEARNING_RATE), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ====================================================================
# 5. TRAINING LOOP
# ====================================================================
def train_system():
    FINAL_FEATURES = cfg.RAW_COLS + ['rain_1h', 'soil_rate', 'soil_acc', 'rain_6h', 'soil_trend_6h', 'rain_24h', 'soil_max_24h', 'rain_x_soil', 'rain_x_geo']
    
    log("Scaling Data...")
    scaler = RobustScaler()
    train_mask = ~df_proc['is_test']
    scaler.fit(df_proc.loc[train_mask, FINAL_FEATURES])
    df_proc[FINAL_FEATURES] = scaler.transform(df_proc[FINAL_FEATURES])
    joblib.dump(scaler, cfg.SCALER_PATH)
    
    results_log = []

    for h in cfg.HORIZONS:
        print(f"\n{'='*60}")
        print(f"🚀 TCN HORIZON: +{h/60:.1f} Hours ({h} mins)")
        print(f"{'='*60}")
        
        X_all, y_all, test_mask = create_sequences(df_proc, FINAL_FEATURES, h)
        
        X_train = X_all[~test_mask]; y_train = y_all[~test_mask]
        X_test  = X_all[test_mask];  y_test  = y_all[test_mask]
        
        # Remove NaNs
        valid_tr = ~np.isnan(y_train); X_train = X_train[valid_tr]; y_train = y_train[valid_tr]
        valid_ts = ~np.isnan(y_test);  X_test = X_test[valid_ts];   y_test = y_test[valid_ts]
        
        # --- SPLIT & BALANCE ---
        # 1. Split Val
        X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
        
        # 2. Balance Train (Over + Down)
        print("⚖️ Balancing Train Data...")
        X_train_bal, y_train_bal = balance_data(X_train, y_train)
        print(f"   Balanced Train counts: {Counter(y_train_bal)}")
        
        # 3. Balance Validation (Optional but good for monitoring)
        # print("⚖️ Balancing Validation Data...")
        # X_val_bal, y_val_bal = balance_data(X_val, y_val) # ถ้า Val น้อยเกินไป ข้ามขั้นตอนนี้ได้
        
        # Encode
        y_train_cat = to_categorical(y_train_bal, 3)
        y_val_cat = to_categorical(y_val, 3)
        
        # Weights
        safety_factor = 1.0 + (h/1440.0)
        cw = {k: v * safety_factor for k, v in cfg.CLASS_WEIGHTS.items()}
        
        # Build & Train
        model = build_tcn_model((cfg.SEQUENCE_LENGTH, len(FINAL_FEATURES)), 3)
        save_path = os.path.join(cfg.MODEL_DIR, f"tcn_h{h}.h5")
        
        cbs = [
            EarlyStopping(patience=12, restore_best_weights=True, monitor='val_loss'),
            ReduceLROnPlateau(patience=5, factor=0.5, monitor='val_loss'),
            ModelCheckpoint(save_path, save_best_only=True, monitor='val_accuracy', verbose=0)
        ]
        
        model.fit(X_train_bal, y_train_cat, validation_data=(X_val, y_val_cat),
                  epochs=cfg.EPOCHS, batch_size=cfg.BATCH_SIZE, 
                  class_weight=cw, callbacks=cbs, verbose=1)
        
        # Evaluate
        probs = model.predict(X_test, verbose=0)
        y_pred = np.argmax(probs, axis=1)
        
        report = classification_report(y_test, y_pred, target_names=['Normal', 'Warning', 'Critical'], output_dict=True, zero_division=0)
        f1_norm = report['Normal']['f1-score']
        f1_warn = report['Warning']['f1-score']
        f1_crit = report['Critical']['f1-score']
        acc = accuracy_score(y_test, y_pred)
        
        print(f"   📊 Accuracy : {acc:.4f}")
        print(f"   🟢 F1-Norm  : {f1_norm:.4f}")
        print(f"   🟡 F1-Warn  : {f1_warn:.4f}")
        print(f"   🔴 F1-Crit  : {f1_crit:.4f}")
        
        results_log.append({'Horizon': h, 'Acc': acc, 'F1_Norm': f1_norm, 'F1_Warn': f1_warn, 'F1_Crit': f1_crit})

    summary = pd.DataFrame(results_log)
    summary.to_csv("./model/tcn_horizon_performance.csv", index=False)
    print("\n=== FINAL SUMMARY ===")
    print(summary)
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(summary['Horizon'], summary['F1_Crit'], marker='o', color='red', label='Critical F1')
    plt.plot(summary['Horizon'], summary['F1_Warn'], marker='s', color='orange', label='Warning F1')
    plt.title("TCN Forecast Performance")
    plt.xlabel("Minutes Ahead")
    plt.ylabel("F1 Score")
    plt.legend(); plt.grid(True)
    plt.show()

if __name__ == "__main__":
    train_system()

✅ Config Loaded.
[INFO] 00:31:06 - Scanning Data...
   + Loaded: final_moving_avg_devID109_test_prepared.csv [TRAIN]
   + Loaded: label_for_dev108_test_prepared.csv [TEST]
   + Loaded: label_for_dev109_test_prepared.csv [TRAIN]
   + Loaded: moving_avg_for_train_prepared.csv [TRAIN]
[INFO] 00:31:07 - Generating Features...
[INFO] 00:31:08 - Scaling Data...

🚀 TCN HORIZON: +0.0 Hours (0 mins)
⚖️ Balancing Train Data...
   Original counts: Counter({np.int64(0): 8731, np.int64(1): 437, np.int64(2): 301})
   Balanced Train counts: Counter({np.int64(1): 5000, np.int64(0): 5000, np.int64(2): 5000})
Epoch 1/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.5863 - loss: 18.5407

235/235 ━━━━━━━━━━━━━━━━━━━━ 30s 82ms/step - accuracy: 0.5864 - loss: 18.4937 - val_accuracy: 0.7348 - val_loss: 0.8822 - learning_rate: 0.0020
Epoch 2/100
234/235 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.6444 - loss: 2.0198

235/235 ━━━━━━━━━━━━━━━━━━━━ 19s 81ms/step - accuracy: 0.6443 - loss: 2.0200 - val_accuracy: 0.7614 - val_loss: 0.9162 - learning_rate: 0.0020
Epoch 3/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 23s 96ms/step - accuracy: 0.6116 - loss: 2.0213 - val_accuracy: 0.6799 - val_loss: 0.8229 - learning_rate: 0.0020
Epoch 4/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 27s 116ms/step - accuracy: 0.6450 - loss: 1.9665 - val_accuracy: 0.6795 - val_loss: 0.7966 - learning_rate: 0.0020
Epoch 5/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 23s 93ms/step - accuracy: 0.6481 - loss: 1.9066 - val_accuracy: 0.6858 - val_loss: 0.8133 - learning_rate: 0.0020
Epoch 6/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 19s 79ms/step - accuracy: 0.6565 - loss: 1.9349 - val_accuracy: 0.7238 - val_loss: 0.7915 - learning_rate: 0.0020
Epoch 7/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 22s 94ms/step - accuracy: 0.6559 - loss: 1.8783 - val_accuracy: 0.7251 - val_loss: 0.6901 - learning_rate: 0.0020
Epoch 8/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 20s 85ms/step - accuracy: 0.6595 - loss:

KeyboardInterrupt: 